# 🦀 Day 1: Async Concepts

Today: futures, `.await`, `async fn` — concurrency without thread overhead.

---

## Why Async?

- **Threads** use OS resources: each thread has its own stack (~1–2 MB)
- **Async** runs many tasks on few threads: cooperative multitasking
- Great for I/O-bound work: network, files, databases — waiting, not CPU crunching

## What is a Future?

A **Future** is a value that will be available *later*. It's a computation that may not have finished yet. When you `.await` it, you're saying: "pause here until the value is ready."

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

In [ ]:
use tokio::runtime::Runtime;

// async fn returns a Future — it doesn't run until awaited
async fn greet() -> String {
    "Hello, async!".to_string()
}

let rt = Runtime::new().unwrap();
let result = rt.block_on(async {
    greet().await
});
println!("{}", result);

## async fn and .await

- `async fn` returns a `Future`, not the inner type directly
- `.await` yields control until the Future completes
- In EvCxR we use `Runtime::new().unwrap().block_on(async { ... })` to run async code

In [ ]:
use tokio::time::{sleep, Duration};

async fn slow_task(name: &str, ms: u64) -> String {
    sleep(Duration::from_millis(ms)).await;
    format!("{} done", name)
}

let rt = Runtime::new().unwrap();
rt.block_on(async {
    // Sequential: one after another
    let a = slow_task("A", 50).await;
    let b = slow_task("B", 30).await;
    println!("{}, {}", a, b);
});

## Async vs Threads

| | Threads | Async |
|---|---|---|
| Scheduling | Preemptive (OS) | Cooperative (runtime) |
| Overhead | Higher (stack per thread) | Lower (tasks share threads) |
| Best for | CPU-bound | I/O-bound |

In [ ]:
// Multiple async calls in sequence vs concurrent (preview)
// Sequential: total time = sum of delays
// Concurrent: total time ≈ max of delays (we'll see tokio::join! next)

let rt = Runtime::new().unwrap();
let (a, b) = rt.block_on(async {
    let a = slow_task("Task1", 100).await;
    let b = slow_task("Task2", 100).await;
    (a, b)
});
println!("{}, {}", a, b);

## 🏋️ Exercise

In [ ]:
// Exercise: Write an async function that simulates fetching user data.
// It should sleep for 100ms, then return a struct or tuple like (id, name).
// Call it from block_on and print the result.

// YOUR CODE HERE

## 🎯 Key Takeaways

1. **Future** = value that will be available later
2. `async fn` returns a Future; `.await` runs it and yields until done
3. Async is **cooperative** — tasks yield at `.await` points
4. Use `Runtime::block_on(async { ... })` in EvCxR to run async code
5. Great for I/O-bound work: network, files, DB

---
**Tomorrow:** Tokio runtime — spawning tasks and running them concurrently! ⚡